## MCP reserved method names (JSON-RPC)

On the wire, MCP is JSON-RPC. The strings below are **reserved method names** from the spec — not GitHub-specific. Python `session.list_tools()` is only a wrapper around `"tools/list"`.

A real host (Claude, Codex) speaks these names. GitHub MCP answers the ones its handshake **capabilities** advertised.

### Lifecycle (every connection)

| Method | Direction | What it is |
|--------|-----------|------------|
| `initialize` | client → server | First request: protocol version, clientInfo, client capabilities |
| `notifications/initialized` | client → server | Handshake done; normal requests may follow |
| `ping` | either way | Liveness check |
| `notifications/cancelled` | either way | Cancel an in-flight request |
| `notifications/progress` | either way | Progress for a long call |

### Tools

| Method | What it is |
|--------|------------|
| `tools/list` | Catalog of tools (name, description, JSON Schema) |
| `tools/call` | Run one tool |
| `notifications/tools/list_changed` | Server says the catalog changed; host should list again |

### Resources

| Method | What it is |
|--------|------------|
| `resources/list` | Concrete URIs you can read now |
| `resources/templates/list` | URI patterns with `{placeholders}` |
| `resources/read` | Fetch one URI (file text, HTML UI, …) |
| `resources/subscribe` / `unsubscribe` | Watch a URI for updates |
| `notifications/resources/list_changed` | Catalog changed |
| `notifications/resources/updated` | A subscribed URI changed |

### Prompts

| Method | What it is |
|--------|------------|
| `prompts/list` | Named recipes + argument schema |
| `prompts/get` | Expand a recipe into messages |
| `notifications/prompts/list_changed` | Catalog changed |

### Other (optional capabilities)

| Method | What it is |
|--------|------------|
| `completion/complete` | Autocomplete a prompt arg or URI template placeholder |
| `logging/setLevel` | Ask the server to emit `notifications/message` at a log level |
| `notifications/message` | Server log line (not the handshake) |

Server → client (this notebook does **not** implement these): `sampling/createMessage`, `roots/list`, `elicitation/create`.

The next cell prints the JSON-RPC **method string** on every request, then explores `ping`, `tools/list`, and `completion/complete`.


In [1]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()

REMOTE_MCP_URL="https://api.githubcopilot.com/mcp"
TOOLSETS="repos, issues, pull_requests, context, users, git"
SERVER_NAME="github"
GITHUB_TOKEN=os.getenv("GITHUB_TOKEN")

### Initializing Github MCP Connection

In [2]:
def build_github_connection()->dict:
    headers={
        "X-MCP-Toolsets": TOOLSETS,
        "X-MCP-Readonly": "true",
        "Authorization": f"Bearer {GITHUB_TOKEN}"
    }
    return {
        SERVER_NAME: {
            "url": REMOTE_MCP_URL,
            "headers": headers,
            "transport": "http",
            "timeout": 60,
            "sse_read_timeout": 300,
        }
    }

In [3]:
CONNECTIONS=build_github_connection()

### Creating MCP Client instance on our Machine/Host

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client=MultiServerMCPClient(CONNECTIONS)

In [5]:
def parse_tools(tool_result):
    rows=[]

    for tool in tool_result.tools:
        schema=getattr(tool, "inputSchema", None) or {}

        # if hasattr(tool, "model_dump"):
        #     schema=schema.model_dump()
        
        rows.append({
            "name": tool.name,
            "description": tool.description,
            "input_schema": schema if isinstance(schema, dict) else {}
        })
    
    return sorted(rows, key=lambda x: x["name"])

### Fetching list of resources and prompts by Github MCP Server

In [ ]:
async def fetch_catalog(client, server_name: str):
    async with client.session(server_name) as session:
        resources=await session.list_resources() # These are the list of resources returned by github mcp server
        prompts=await session.list_prompts() # These are the list of prompts returned by github mcp server
    
    return resources, prompts

In [53]:
resources, prompts=await fetch_catalog(mcp_client, SERVER_NAME)
print(resources.resources)
print(prompts.prompts)

[Resource(name='get_me_ui', title=None, uri=AnyUrl('ui://github-mcp-server/get-me'), description='MCP App UI for the get_me tool', mimeType='text/html;profile=mcp-app', size=None, icons=None, annotations=None, meta=None)]
[Prompt(name='AssignCodingAgent', title=None, description='Assign GitHub Coding Agent to multiple tasks in a GitHub repository.', arguments=[PromptArgument(name='repo', description='The repository to assign tasks in (owner/repo).', required=True)], icons=[Icon(src='', mimeType='image/png', sizes=None, theme='light'), Icon(src='', mimeType='image/png', sizes=None, theme='dark')], meta=None), Prompt(name='issue_to_fix_workflow', title=None, description='Create an issue for a problem and then generate a pull request to fix it', arguments=[PromptArgument(name='owner', description='Repository owner', required=True), PromptArgument(name='repo', description='Repository name', required=True), PromptArgument(name='title', description='Issue title', required=True), PromptArgume

### Fetching list of tools returned by Github MCP Server

In [39]:
tools=await mcp_client.get_tools(server_name=SERVER_NAME)

In [40]:
tools

[StructuredTool(name='get_commit', description='Get details for a commit from a GitHub repository', args_schema={'type': 'object', 'properties': {'detail': {'type': 'string', 'description': 'Level of detail to include for changed files. "none" omits stats and files entirely. "stats" (default) includes per-file metadata: filename, status, and lines-of-code counts (additions, deletions, changes), with no patch content. "full_patch" additionally includes the unified diff content for each file and can be very large.', 'default': 'stats', 'enum': ['none', 'stats', 'full_patch']}, 'owner': {'type': 'string', 'description': 'Repository owner', 'x-mcp-header': 'owner'}, 'page': {'type': 'number', 'description': 'Page number for pagination (min 1)', 'minimum': 1}, 'perPage': {'type': 'number', 'description': 'Results per page for pagination (min 1, max 100)', 'minimum': 1, 'maximum': 100}, 'repo': {'type': 'string', 'description': 'Repository name', 'x-mcp-header': 'repo'}, 'sha': {'type': 'str

### Creating Agent instance

In [25]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent

In [30]:
model=ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY"),
)

### Creating Agent and passing the list of tools and configs

In [41]:
SYSTEM_PROMPT="""
You are a github assistant. Use MCP tools for live data.
Do not create any PRs or repos.
"""

agent=create_agent(
    model=model,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

### Invoking the agent Asynchronously since its a Remote MCP Server

In [44]:
question="Who am I on github? Name a few repositories if available."

output=await agent.ainvoke(
    {
        "messages": [
            {"role": "user", "content": question},
        ]
    }
)

In [45]:
print(output["messages"][-1].content)

**GitHub profile**

- **Username / login:** `Shubhii-A2Z`  
- **Name:** Shubham Negi  
- **Company:** Delhi Technological University  
- **Bio:** “SWE Intern 26' @Google – Pre‑Final Year”  
- **Public repos:** 32 (plus several private repos)  
- **Followers:** 4 | **Following:** 3  
- **Account created:** 16 Feb 2024  

**A few of your repositories (as they appear publicly)**  

| Repo | Description / language | Public/Private | Link |
|------|------------------------|----------------|------|
| **ProjectX-Frontend** | Front‑end for the upcoming “ProjectX” (TypeScript) | Public | <https://github.com/Shubhii-A2Z/ProjectX-Frontend> |
| **Model-Context-Protocol** | Jupyter‑notebook based exploration of a model‑context protocol | Public | <https://github.com/Shubhii-A2Z/Model-Context-Protocol> |
| **MultiStaged-Docker** | Docker‑file examples for multi‑stage builds | Private (you can see it because you own it) | <https://github.com/Shubhii-A2Z/MultiStaged-Docker> |
| **Parking-Lot-LLD** | L